In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import pandas as pd
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from sklearn.cluster import DBSCAN, HDBSCAN

import sys

sys.path.append("../..")

from src import IOFunctions


from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20260320_162728.log
/tmp/ipykernel_2754302/1679663959.py:26: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter(dark_background=False)


In [2]:
R, G, B, wavelength = S_F.getpixelefficiency()

In [3]:
pixel_QYs = np.vstack([B, G, R])

In [4]:
dyes = ["Alexa Fluor 488", "Cy3"]

In [5]:
spectra = S_F.get_dye_or_filter_data(names=dyes, wavelength=wavelength)

In [ ]:
cy3 = spectra[0, :]
cy5 = spectra[1, :]

In [ ]:
E_param = np.linspace(0, 1, 1000)


In [ ]:
colours = np.ones([4, len(E_param)])
c1 = np.array([1, 0.6, 0.333])
c2 = np.array([1, 0.333, 0.333])

In [ ]:
pixel_efficiencies = np.zeros([3, len(E_param)])
for i, Ev in enumerate(E_param):
    spectrum = cy3 * (1 - Ev) + cy5 * (Ev)
    spectrum = spectrum / np.sum(spectrum)
    _, PE = S_F.get_pixel_fractions_rawspectra(
        spectra=spectrum, wavelength=wavelength, pixel_QYs=pixel_QYs
    )
    pixel_efficiencies[:, i] = PE / np.sum(PE)
    colours[:3, i] = c1 * (1 - Ev) + c2 * (Ev)

In [ ]:
pixel_efficiencies.shape

In [ ]:
x = np.linspace(0, 2, 1000)
E = 1 / (1 + np.power(x / 1, 6.0))

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 0.5], height=2)


axs[0] = plotter.line_plot(
    axs[0], x, E, xaxislabel=r"R/R$_0$", yaxislabel="FRET Efficiency (E)", color="white"
)
axs[0].set_ylim([-0.01, 1.02])


from mpl_toolkits.axes_grid1.inset_locator import inset_axes

inset_ax = inset_axes(
    axs[0],
    width="35%",  # width = 30% of parent_bbox
    height="35%",  # height : 1 inch
    loc=1,
)
for i in np.linspace(0.1, 0.9, 9):
    inset_ax.plot(wavelength, cy3 * (1 - i), color="#ff9955", lw=0.5)
    inset_ax.plot(wavelength, cy5 * (i), color="#ff5555", lw=0.5)
inset_ax.set_yticklabels([])
inset_ax.set_xlim([500, 750])
inset_ax.set_ylim([0, 0.028])
inset_ax.set_xticks([500, 600, 700])
inset_ax.set_xlabel("wavelength/nm", fontsize=6)
inset_ax.set_ylabel("fluo/norm", fontsize=6)

axs[1] = plotter.line_plot(
    axs[1], E[::-1], pixel_efficiencies[-1, ::-1], xaxislabel=r"E", yaxislabel="camera detection", color='#99ff55',
)

axs[1] = plotter.line_plot(
    axs[1], E[::-1], pixel_efficiencies[1, ::-1], xaxislabel=r"E", yaxislabel="camera detection", color='#ff2a2a',
)


axs[1] = plotter.line_plot(
    axs[1], E[::-1], pixel_efficiencies[0, ::-1], xaxislabel=r"E", yaxislabel="relative pixel QE", color='#5555ff',
)
axs[1].set_xlim([0, 1])
axs[1].set_ylim([0, 1])

folder = "/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Conference Presentations/20260302_DurhamTalks"

plt.savefig(os.path.join(folder, "example_FRET_colours.svg"), dpi=600, format="svg")
plt.show()